In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
import os, sys, subprocess

REPO_URL  = "https://github.com/biswajit-nag/Predict-Customer-Churn.git"
REPO_ROOT = "/kaggle/working/Predict-Customer-Churn"

if not os.path.exists(REPO_ROOT):
    subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)

os.chdir(REPO_ROOT)                       # CWD = repo root (fixes data paths + git_info)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)         # makes `from src.xxx import ...` resolve
print("CWD:", os.getcwd())


Cloning into '/kaggle/working/Predict-Customer-Churn'...


CWD: /kaggle/working/Predict-Customer-Churn


In [3]:
!pip install -q pytabkit

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())   # expect 2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12

In [4]:
import pytabkit, torch
print(torch.__version__)
print("CUDA:", torch.cuda.is_available())
# confirm pytabkit's key class is reachable
from pytabkit import TabM_D_Classifier
print("TabM_D_Classifier imported OK")


2.10.0+cu128
CUDA: True
TabM_D_Classifier imported OK


In [5]:
import shutil
from pathlib import Path

raw_dir = Path(REPO_ROOT) / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
for f in ("train.csv", "test.csv"):
    shutil.copy(f"/kaggle/input/competitions/playground-series-s6e3/{f}", raw_dir / f)

from src.data import prepare_data
train_df, test_df = prepare_data(force=True)   # writes data/processed/*.parquet


Preprocessed and saved: train_df (594194, 42), test_df (254655, 41)


In [6]:
import json, numpy as np, pandas as pd, joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

from src.tracking import PROJECT_ROOT, DATA_DIR, RUNS_DIR, load_runs, delete_run
from src.cv import run_cv_experiment, save_experiment


### Feature engineering

`engineer_features(df)` is the per-experiment hook for adding features. The
output is cached to `data/processed/train_df_{DATA_VERSION}.parquet` so future
runs reload it instantly — and so any past run is reconstructible by reading
the parquet whose suffix matches that run's `data_version` field.

**Strict no-leakage rule**: only row-wise (stateless) transforms here. Anything
that needs to be *fitted* on training data — target encoding, scaling,
imputation, frequency counts, or any aggregate over rows — would leak val/test
information across folds if computed once on the full training set. Those
must be done inside the fold loop instead.

When you change `engineer_features`, **bump `DATA_VERSION`** so a new parquet
is written rather than the stale cache being reused.

In [7]:
import pyarrow as pa
import pyarrow.parquet as pq

DATA_VERSION = 'fe_v0'  # bump whenever engineer_features changes — see rule below


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Row-wise (stateless) feature engineering.

    Each new column must depend ONLY on values from the row being transformed.
    No aggregates, no fitted encoders, no cross-row statistics — those leak
    val/test information into training when applied before the CV split, so
    they belong inside the fold loop instead.

    Examples of acceptable transforms:
        df['x_per_y']     = df['x'] / (df['y'] + 1)
        df['log_x']       = np.log1p(df['x'])
        df['x_above_med'] = (df['x'] > 100).astype(int)
    """
    df = df.copy()
    # Add row-wise features here.
    return df


# Cache engineered parquets per DATA_VERSION. Past runs are reconstructible by
# loading the parquet whose suffix matches that run's logged data_version.
fe_train_path = DATA_DIR / f'train_df_{DATA_VERSION}.parquet'
fe_test_path  = DATA_DIR / f'test_df_{DATA_VERSION}.parquet'

if fe_train_path.exists() and fe_test_path.exists():
    train_df = pd.read_parquet(fe_train_path)
    test_df  = pd.read_parquet(fe_test_path)
    print(f'Loaded cached FE: {DATA_VERSION}')
else:
    train_df = engineer_features(train_df)
    test_df  = engineer_features(test_df)
    pq.write_table(pa.Table.from_pandas(train_df, preserve_index=False), fe_train_path)
    pq.write_table(pa.Table.from_pandas(test_df,  preserve_index=False), fe_test_path)
    print(f'Computed and cached FE: {DATA_VERSION}')

Loaded cached FE: fe_v0


In [8]:
# Refresh feature list and design matrices from the engineered dataframes.
encoded_features = [c for c in train_df.columns if c not in ('id', 'Churn')]
X_train = train_df[encoded_features]
y_train = train_df['Churn']
X_test  = test_df[encoded_features]
print(f'X_train: {X_train.shape}  X_test: {X_test.shape}  features: {len(encoded_features)}')

X_train: (594194, 40)  X_test: (254655, 40)  features: 40


### Run configuration — TabM baseline

[TabM](https://github.com/yandex-research/tabm) (Gorishniy et al., 2024) is an
MLP that cheaply simulates a *deep ensemble*: it trains `k` submodels that share
almost all weights but carry rank-1 per-member adapters (BatchEnsemble), then
averages their predictions. This gives ensemble-level accuracy at close to
single-MLP cost, and is frequently competitive with GBDTs on tabular data.

**Integration notes** (read before running):
- TabM is PyTorch-based. The CV harness needs a `.fit` / `.predict_proba`
  estimator, so the cell below uses [`pytabkit`](https://github.com/dholzmueller/pytabkit)'s
  sklearn wrapper. Neither `pytabkit` nor `torch` is currently a project
  dependency — add with `uv add pytabkit` first. **Verify the class name and
  kwargs against the installed version**; the scaffold is a best guess.
- `requires-python = ">=3.14"` here; confirm a PyTorch wheel exists for your
  Python version before assuming the install will succeed.
- Unlike GBDTs, MLPs need scaled inputs. `pytabkit` handles preprocessing
  internally, so the existing one-hot matrix can be fed as-is.

In [9]:
# Requires: uv add pytabkit   (pulls in PyTorch)
# VERIFY the class name and kwargs against your installed pytabkit version —
# the API below is a best-guess scaffold, not confirmed against this environment.
from pytabkit import TabM_D_Classifier

# tabm_params = {
#     'device':       'cpu',   # use 'cpu' if no GPU — but CPU on ~600k rows is slow
#     'random_state': 42,
#     'n_epochs':     100,      # early stopping typically halts well before this
#     'batch_size':   1024,     # > the 256 paper default, for throughput on a large set
# }

tabm_params = {'device': 'cuda', 'random_state': 42, 'n_epochs': 100, 'batch_size': 1024}


run_config = {
    'model_factory': lambda params: TabM_D_Classifier(**params),
    'params':        tabm_params,
    'metric':        accuracy_score,
    'metric_name':   'accuracy',
    'cv':            StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'tag':           'tabm-baseline',
    'notes': (
    'TabM (pytabkit TabM_D_Classifier, default architecture) on Kaggle T4 GPU. '
    'torch=2.10.0+cu128, pytabkit=1.7.3. '
    'Data regenerated on-platform from competition CSVs — data_hash will differ from local runs. '
    'GPU non-determinism means OOF score is not bit-reproducible locally.'
),
    'parent_run_id': '',
    'save_models':   False,   # torch-backed; skip joblib dump for the first run
    'data_version':  DATA_VERSION,
}

In [10]:
# Step 1 — Run the experiment.
# Fits models fold by fold, prints scores as they complete, then prints the
# final OOF metric. Nothing is written to disk yet.
# Full implementation: src/cv.py
result = run_cv_experiment(run_config, X_train, y_train, X_test, encoded_features)

Run ID: 20260601-022539-10bd38
Tag:    tabm-baseline



/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Fold 0: accuracy=0.8576  roc_auc=0.9131  (fit 651.2s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Fold 1: accuracy=0.8588  roc_auc=0.9143  (fit 1732.9s)


...

Fold 2: accuracy=0.8586  roc_auc=0.9137  (fit 724.0s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Fold 3: accuracy=0.8596  roc_auc=0.9149  (fit 764.3s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Fold 4: accuracy=0.8580  roc_auc=0.9120  (fit 802.0s)

OOF accuracy: 0.8585
OOF ROC-AUC:  0.9136
Folds:        0.8585 ± 0.0007

Run complete. Call save_experiment(result) to log this run permanently.


In [11]:
# Step 2 — Save the run (optional).
# Review the fold scores and OOF metric printed above, then run this cell
# to permanently log the run to experiments/runs.csv and write its artifact
# directory. Skip this cell to discard the run without any trace on disk.
run_id = save_experiment(result)

Saved to: /kaggle/working/Predict-Customer-Churn/experiments/runs/20260601-022539-10bd38
